## Lab 1: Data Representation

Dianne Yumol

In [1]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, make_scorer
import warnings
warnings.filterwarnings("ignore")
%matplotlib inline

# Try to import your module (preferred). If not present, we'll raise an informative error.
try:
    from variance_pipeline_no_eqd import (
        AutomatedVarianceThreshold,
        SimpleMinMaxScaler, SimpleZScoreScaler, SimpleRobustScaler, Imputer
    )
except Exception as e:
    raise ImportError("Could not import variance_pipeline_no_eqd. Make sure the file is in the working directory.") from e

print("Imports OK. Module loaded.")


Imports OK. Module loaded.


In [2]:
telco_path = "telco_customer_churn.csv"
if not os.path.exists(telco_path):
    raise FileNotFoundError("telco_customer_churn.csv not found in working directory. Upload or place it here to proceed.")

# 1) load
df = pd.read_csv(telco_path)

# 2) ensure TotalCharges numeric
if 'TotalCharges' in df.columns:
    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# 3) drop common id columns if present
for idcol in ['customerID', 'CustomerID', 'id', 'Id']:
    if idcol in df.columns:
        df = df.drop(columns=[idcol])

# 4) identify categorical columns (exclude target 'Churn' if present)
exclude = ['Churn']
cat_cols = [c for c in df.select_dtypes(include=['object','category','bool']).columns if c not in exclude]

# 5) impute numeric columns (median) and categorical (most_frequent)
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
num_imputer = Imputer(strategy="median", columns=numeric_cols)
df = num_imputer.fit_transform(df)

if cat_cols:
    cat_imputer = Imputer(strategy="most_frequent", columns=cat_cols)
    df = cat_imputer.fit_transform(df)

# 6) one-hot encode categorical columns
df_ohe = pd.get_dummies(df, columns=cat_cols, drop_first=False, dtype=int)

# 7) final safety fill if any NaNs remain
if df_ohe.isna().values.any():
    df_ohe = df_ohe.fillna(df_ohe.median(numeric_only=True))

# Confirm
print("Preprocessed Telco shape:", df_ohe.shape)
print("Numeric columns count:", len([c for c in df_ohe.columns if df_ohe[c].dtype.kind in "fi"]))
print("Sample columns:", df_ohe.columns[:20].tolist())


Preprocessed Telco shape: (7043, 46)
Numeric columns count: 45
Sample columns: ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges', 'Churn', 'gender_Female', 'gender_Male', 'Partner_No', 'Partner_Yes', 'Dependents_No', 'Dependents_Yes', 'PhoneService_No', 'PhoneService_Yes', 'MultipleLines_No', 'MultipleLines_No phone service', 'MultipleLines_Yes', 'InternetService_DSL', 'InternetService_Fiber optic', 'InternetService_No', 'OnlineSecurity_No']


In [3]:
scalers = {
    'minmax': SimpleMinMaxScaler(feature_range=(0.0, 1.0)),
    'zscore': SimpleZScoreScaler(),
    'robust': SimpleRobustScaler()
}

methods = ['percentile', 'elbow', 'information_theory', 'emc_inspired']
# include cross_validation if Churn present in dataset
has_churn = 'Churn' in df_ohe.columns
if has_churn:
    methods.append('cross_validation')

print("Scalers:", list(scalers.keys()))
print("Methods:", methods)


Scalers: ['minmax', 'zscore', 'robust']
Methods: ['percentile', 'elbow', 'information_theory', 'emc_inspired', 'cross_validation']


In [4]:
# Prepare label vector if present
y = None
if has_churn:
    # convert Churn to binary 0/1 if it's strings like 'Yes'/'No'
    y_raw = df_ohe['Churn']
    if y_raw.dtype.kind in "O":
        y = (y_raw.astype(str).str.lower() == 'yes').astype(int).to_numpy()
    else:
        # numeric already
        y = y_raw.to_numpy().astype(int)
    print("Churn label distribution:", np.bincount(y))

# We'll store rows of results
rows = []

# numeric columns to consider (exclude Churn)
numeric_cols = [c for c in df_ohe.columns if df_ohe[c].dtype.kind in "fi" and c != 'Churn']
print("Number of numeric columns considered:", len(numeric_cols))

# For each scaler, compute scaled numeric_df and per-column variances
for scaler_name, scaler in scalers.items():
    print("\nRunning scaler:", scaler_name)
    Xnum = df_ohe[numeric_cols].to_numpy(dtype=float)
    scaler.fit(Xnum)
    Xs = scaler.transform(Xnum)
    scaled_numeric_df = pd.DataFrame(Xs, columns=numeric_cols, index=df_ohe.index)
    # combined numeric for variance computation (only numeric)
    numeric_combined = scaled_numeric_df  # we only need numeric variances
    var_series = numeric_combined.var(axis=0, ddof=0)
    total_variance = var_series.sum()
    # For each method, compute selector on numeric_combined (pass y if supervised)
    for method in methods:
        selector = AutomatedVarianceThreshold(method=method)
        try:
            selector.fit(numeric_combined.to_numpy(), y if method=='cross_validation' else None)
            mask = selector.get_selected_features_mask()
            if mask is None:
                raise RuntimeError("Selector returned no mask")
            kept_indices = np.where(mask)[0]
            kept_cols = [numeric_combined.columns[i] for i in kept_indices]
            kept_count = len(kept_cols)
            # fraction variance kept
            if total_variance > 0 and kept_count > 0:
                frac_var_kept = float(var_series.loc[kept_cols].sum() / total_variance)
            else:
                frac_var_kept = np.nan
            # Prepare CV evaluation on pruned features if Churn present
            acc_mean = np.nan
            auc_mean = np.nan
            if has_churn and kept_count >= 2:
                X_pruned = numeric_combined.loc[:, kept_cols].to_numpy()
                # accuracy with logistic regression
                clf = LogisticRegression(max_iter=1000, solver='liblinear')
                skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
                try:
                    acc = cross_val_score(clf, X_pruned, y, cv=skf, scoring='accuracy')
                    acc_mean = float(np.mean(acc))
                except Exception as e:
                    acc_mean = np.nan
                # AUC using cross_val_score with 'roc_auc' scoring
                try:
                    auc = cross_val_score(clf, X_pruned, y, cv=skf, scoring='roc_auc')
                    auc_mean = float(np.mean(auc))
                except Exception as e:
                    # fall back: compute cross-validated probabilities manually if needed
                    auc_mean = np.nan
            rows.append({
                'scaler': scaler_name,
                'method': method,
                'threshold': selector.get_threshold(),
                'kept_count': kept_count,
                'total_numeric': len(numeric_combined.columns),
                'fraction_variance_kept': frac_var_kept,
                'accuracy_cv': acc_mean,
                'auc_cv': auc_mean
            })
            print(f"  method={method:17} kept={kept_count:2d} frac_var={frac_var_kept:0.3f} acc={acc_mean} auc={auc_mean}")
        except Exception as e:
            print(f"  method={method:17} ERROR: {e}")
            rows.append({
                'scaler': scaler_name,
                'method': method,
                'threshold': None,
                'kept_count': None,
                'total_numeric': len(numeric_combined.columns),
                'fraction_variance_kept': None,
                'accuracy_cv': None,
                'auc_cv': None,
                'error': str(e)
            })

# Compile DataFrame
df_results = pd.DataFrame(rows)

# Ensure all expected columns exist
expected_cols = ['scaler','method','threshold','kept_count','total_numeric',
                 'fraction_variance_kept','accuracy_cv','auc_cv','error']
for col in expected_cols:
    if col not in df_results.columns:
        df_results[col] = np.nan

# Reorder
df_results = df_results[expected_cols]

# Clean formatting
df_results = df_results.fillna(value=np.nan)
df_results.style.format({
    'threshold': '{:.6g}',
    'fraction_variance_kept': '{:.3f}',
    'accuracy_cv': '{:.3f}',
    'auc_cv': '{:.3f}'
})


Churn label distribution: [5174 1869]
Number of numeric columns considered: 45

Running scaler: minmax
  method=percentile        kept= 3 frac_var=0.085 acc=0.7346301575908123 auc=0.6974472690928675
  method=elbow             kept= 1 frac_var=0.028 acc=nan auc=nan
  method=information_theory kept=18 frac_var=0.498 acc=0.7836121483644105 auc=0.8223561080694672
  method=emc_inspired      kept= 3 frac_var=0.035 acc=0.7346301575908123 auc=0.5611827322465773
  method=cross_validation  kept=24 frac_var=0.646 acc=0.7870196262662107 auc=0.8258542607026265

Running scaler: zscore
  method=percentile        kept= 3 frac_var=0.067 acc=0.7309387904058327 auc=0.7048571027625075
  method=elbow             kept= 3 frac_var=0.067 acc=0.7309387904058327 auc=0.7048571027625075
  method=information_theory kept=45 frac_var=1.000 acc=0.8049088852506614 auc=0.8450870645442873
  method=emc_inspired      kept= 3 frac_var=0.067 acc=0.7346301575908123 auc=0.6270532012934629
  method=cross_validation  kept=29 fr

,scaler,method,threshold,kept_count,total_numeric,fraction_variance_kept,accuracy_cv,auc_cv,error
0,minmax,percentile,0.249972,3,45,0.085,0.735,0.697,nan
1,minmax,elbow,0.249989,1,45,0.028,nan,nan,nan
2,minmax,information_theory,0.231415,18,45,0.498,0.784,0.822,nan
3,minmax,emc_inspired,0.087457,3,45,0.035,0.735,0.561,nan
4,minmax,cross_validation,0.209835,24,45,0.646,0.787,0.826,nan
5,zscore,percentile,1,3,45,0.067,0.731,0.705,nan
6,zscore,elbow,1,3,45,0.067,0.731,0.705,nan
7,zscore,information_theory,1,45,45,1.000,0.805,0.845,nan
8,zscore,emc_inspired,1,3,45,0.067,0.735,0.627,nan
9,zscore,cross_validation,1,29,45,0.644,0.793,0.836,nan


In [5]:
# Display and save results
display(df_results)
df_results.to_csv("telco_variance_threshold_results_summary.csv", index=False)
print("Saved summary to telco_variance_threshold_results_summary.csv")

,scaler,method,threshold,kept_count,total_numeric,fraction_variance_kept,accuracy_cv,auc_cv,error
0,minmax,percentile,0.249972,3,45,0.084536,0.734630,0.697447,NaN
1,minmax,elbow,0.249989,1,45,0.028180,NaN,NaN,NaN
2,minmax,information_theory,0.231415,18,45,0.497691,0.783612,0.822356,NaN
3,minmax,emc_inspired,0.087457,3,45,0.035031,0.734630,0.561183,NaN
4,minmax,cross_validation,0.209835,24,45,0.646470,0.787020,0.825854,NaN
5,zscore,percentile,1.000000,3,45,0.066667,0.730939,0.704857,NaN
6,zscore,elbow,1.000000,3,45,0.066667,0.730939,0.704857,NaN
7,zscore,information_theory,1.000000,45,45,1.000000,0.804909,0.845087,NaN
8,zscore,emc_inspired,1.000000,3,45,0.066667,0.734630,0.627053,NaN
9,zscore,cross_validation,1.000000,29,45,0.644444,0.792842,0.835627,NaN


Saved summary to telco_variance_threshold_results_summary.csv
